In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

## a) Tabla con un renglón por estación-hora y una columna por contaminante


In [2]:
df = pd.read_csv('contaminantes_N5//contaminantes_2026.csv')

# Pivotar por fecha y estación
piv = df.pivot_table(index=['date', 'id_station'], columns='id_parameter', values='valor').reset_index()

print(f"Dimensiones de la tabla pivotada: {piv.shape[0]} renglones y {piv.shape[1]} columnas.")
print("Columnas:", list(piv.columns))
piv.head()

Dimensiones de la tabla pivotada: 33982 renglones y 11 columnas.
Columnas: ['date', 'id_station', 'CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 'PMCO', 'SO2']


id_parameter,date,id_station,CO,NO,NO2,NOX,O3,PM10,PM2.5,PMCO,SO2
0,2026-01-01 00:00:00,AJM,0.84,2.0,26.0,28.0,19.0,67.0,53.0,14.0,28.0
1,2026-01-01 00:00:00,ATI,0.80,2.0,21.0,23.0,18.0,90.0,NaN,NaN,22.0
2,2026-01-01 00:00:00,BJU,1.22,NaN,NaN,NaN,17.0,62.0,41.0,21.0,NaN
3,2026-01-01 00:00:00,CAM,NaN,7.0,44.0,52.0,4.0,57.0,49.0,8.0,25.0
4,2026-01-01 00:00:00,CCA,1.27,1.0,32.0,33.0,14.0,NaN,NaN,NaN,3.0


## b) Covarianza y correlación de CO con NOX. ¿Son distintas?

In [3]:
cov_co_nox = piv['CO'].cov(piv['NOX'])
corr_co_nox = piv['CO'].corr(piv['NOX'])

print(f"Covarianza CO con NOX: {cov_co_nox:.4f}")
print(f"Correlación CO con NOX: {corr_co_nox:.4f}")

Covarianza CO con NOX: 22.3069
Correlación CO con NOX: 0.8568


### ¿Son distintas la covarianza y la correlación?
   - La covarianza solo nos dice la dirección de la asociación (positiva).
   - La correlación nos permite cuantificar la fuerza de la relación lineal: un valor de **$0.8568$** indica una correlación lineal positiva muy fuerte entre el monóxido de carbono y los óxidos de nitrógeno, ya que ambos son subproductos directos de las emisiones de combustión vehicular.

## c) Eigenvalores de la matriz de correlación, con y sin faltantes

In [4]:
pollutants = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PM2.5', 'PMCO', 'SO2']

# 1. Matriz de correlación con faltantes (pairwise)
corr_pairwise = piv[pollutants].corr()
eig_pairwise = np.sort(np.linalg.eigvals(corr_pairwise.values))[::-1]

# 2. Matriz de correlación sin faltantes (complete cases)
df_clean = piv[pollutants].dropna()
corr_clean = df_clean.corr()
eig_clean = np.sort(np.linalg.eigvals(corr_clean.values))[::-1]

eig_df = pd.DataFrame({
    'Componente': [f'λ_{i+1}' for i in range(len(pollutants))],
    'Con faltantes': eig_pairwise,
    'Sin faltantes': eig_clean
})
print(f"Observaciones completas para sin faltantes: {len(df_clean)}")
eig_df

Observaciones completas para sin faltantes: 5809


,Componente,Con faltantes,Sin faltantes
0,λ_1,4.144695,4.188021
1,λ_2,1.921354,1.939947
2,λ_3,1.030131,1.049406
3,λ_4,0.749362,0.806155
4,λ_5,0.535747,0.519994
5,λ_6,0.471860,0.371747
6,λ_7,0.155245,0.124379
7,λ_8,0.000290,0.000282
8,λ_9,-0.008685,0.000069


Notemos que el ultimo eigenvalor con faltantes es negativo a diferencia de sin faltantes que son todos positivos.

## d) Predecir PM2.5 a partir de los demás contaminantes, luego contra solo gases. Calcular R² y VIF.

In [5]:
# 1. PM2.5 contra todos los demás contaminantes
all_preds = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'PM10', 'PMCO', 'SO2']
data_all = piv.dropna(subset=['PM2.5'] + all_preds).copy()

X1 = data_all[all_preds]
X1_const = sm.add_constant(X1)
y1 = data_all['PM2.5']

model1 = sm.OLS(y1, X1_const).fit()

vif1 = pd.DataFrame({
    'Variable': all_preds,
    'VIF': [variance_inflation_factor(X1.values, i) for i in range(len(all_preds))]
})

print(f"--- Modelo 1: PM2.5 contra todos (N = {len(data_all)}) ---")
print(f"R²: {model1.rsquared:.4f}")
print("VIFs:")
print(vif1.to_string(index=False))

--- Modelo 1: PM2.5 contra todos (N = 5809) ---
R²: 0.9988
VIFs:
Variable          VIF
      CO    19.071197
      NO  6487.688169
     NO2  4780.866617
     NOX 18258.517789
      O3     2.434375
    PM10    15.898067
    PMCO     7.771404
     SO2     1.534985


In [6]:
# 2. PM2.5 solo contra gases
gases = ['CO', 'NO', 'NO2', 'NOX', 'O3', 'SO2']
data_gases = piv.dropna(subset=['PM2.5'] + gases).copy()

X2 = data_gases[gases]
X2_const = sm.add_constant(X2)
y2 = data_gases['PM2.5']

model2 = sm.OLS(y2, X2_const).fit()

vif2 = pd.DataFrame({
    'Variable': gases,
    'VIF': [variance_inflation_factor(X2.values, i) for i in range(len(gases))]
})

print(f"--- Modelo 2: PM2.5 solo con gases (N = {len(data_gases)}) ---")
print(f"R²: {model2.rsquared:.4f}")
print("VIFs:")
print(vif2.to_string(index=False))

--- Modelo 2: PM2.5 solo con gases (N = 9247) ---
R²: 0.2665
VIFs:
Variable          VIF
      CO    14.106534
      NO  6597.124872
     NO2  4271.169491
     NOX 17293.175975
      O3     1.705635
     SO2     1.404847


## e) H con una estación y una semana: comprueben H² = H y traza = p. Luego el leverage de todos sin construir H

In [7]:
# 1. H con una estación y una semana
piv['date_dt'] = pd.to_datetime(piv['date'])
sem1 = piv[(piv['date_dt'] >= '2026-01-01') & (piv['date_dt'] <= '2026-01-07 23:59:59')]
st_data = sem1[sem1['id_station'] == 'MER'].dropna(subset=['PM2.5'] + all_preds).copy()

X_sub = sm.add_constant(st_data[all_preds])
X_mat = X_sub.values
n_sub, p_sub = X_mat.shape

# Matriz H: H = X (X' X)^(-1) X'
XtX_inv = np.linalg.pinv(X_mat.T @ X_mat)
H = X_mat @ XtX_inv @ X_mat.T

# Comprobaciones
diff_H2_H = np.max(np.abs(H @ H - H))
trace_H = np.trace(H)

print(f"Dimensiones de H (MER, Semana 1): {H.shape[0]} x {H.shape[1]}")
print(f"Número de parámetros p (incluye constante): {p_sub}")
print(f"1. Idempotencia (H² = H): max|H² - H| = {diff_H2_H:.2e} -> ¿Es H² == H? {np.allclose(H @ H, H)}")
print(f"2. Traza de H = {trace_H:.4f} -> ¿Es Traza == p? {np.isclose(trace_H, p_sub)}")

Dimensiones de H (MER, Semana 1): 144 x 144
Número de parámetros p (incluye constante): 9
1. Idempotencia (H² = H): max|H² - H| = 7.20e-13 -> ¿Es H² == H? True
2. Traza de H = 9.0000 -> ¿Es Traza == p? True


In [8]:
# 2. Leverage de todas las observaciones sin construir H
# Usamos el modelo completo de PM2.5 (N = 5809)
X_full = X1_const.values
XtX_full_inv = np.linalg.pinv(X_full.T @ X_full)

# El leverage h_ii es el i-ésimo elemento diagonal de H: x_i' (X'X)^(-1) x_i
# Lo calculamos eficientemente por filas sin crear la matriz N x N:
leverage_direct = np.sum((X_full @ XtX_full_inv) * X_full, axis=1)

# Verificamos contra statsmodels
infl1 = model1.get_influence()
leverage_sm = infl1.hat_matrix_diag

print(f"Total de observaciones N: {len(X_full)}")
print(f"¿El cálculo directo coincide exactamente con statsmodels? {np.allclose(leverage_direct, leverage_sm)}")
print(f"Suma de todos los leverages (debe ser p = {X_full.shape[1]}): {np.sum(leverage_direct):.4f}")

Total de observaciones N: 5809
¿El cálculo directo coincide exactamente con statsmodels? True
Suma de todos los leverages (debe ser p = 9): 9.0000


## f) Cook: ¿los 10 más influyentes son los de más leverage?

In [9]:
# Distancia de Cook
cooks_d = infl1.cooks_distance[0]

# Top 10 más influyentes según Cook y Top 10 de mayor leverage
top10_cook_idx = np.argsort(cooks_d)[-10:][::-1]
top10_lev_idx = np.argsort(leverage_direct)[-10:][::-1]

comparacion = pd.DataFrame({
    'Ranking': range(1, 11),
    'Top 10 Cook (Más Influyentes)': top10_cook_idx,
    'Distancia Cook': cooks_d[top10_cook_idx],
    'Top 10 Leverage (Puntos Palanca)': top10_lev_idx,
    'Valor Leverage (h_ii)': leverage_direct[top10_lev_idx]
})

print(comparacion.to_string(index=False))

interseccion = set(top10_cook_idx).intersection(set(top10_lev_idx))
print(f"\nObservaciones en común entre ambos grupos: {len(interseccion)} ({list(interseccion)})")

 Ranking  Top 10 Cook (Más Influyentes)  Distancia Cook  Top 10 Leverage (Puntos Palanca)  Valor Leverage (h_ii)
       1                           2680        0.080090                                41               0.038528
       2                             41        0.028160                                35               0.035228
       3                           3175        0.027530                                52               0.034109
       4                           4339        0.024083                              4958               0.033662
       5                           3851        0.019429                                46               0.032561
       6                           3195        0.014434                              3400               0.026711
       7                           3429        0.013981                                58               0.024248
       8                             19        0.012447                                29       

In [10]:
# Reajuste del modelo sin los 10 puntos más influyentes de Cook
mask_keep = np.ones(len(data_all), dtype=bool)
mask_keep[top10_cook_idx] = False

model1_reajustado = sm.OLS(y1.iloc[mask_keep], X1_const.iloc[mask_keep]).fit()

print("--- Comparación de Modelos ---")
print(f"R² Original:     {model1.rsquared:.6f}")
print(f"R² Reajustado:   {model1_reajustado.rsquared:.6f}")
print(f"MSE Residual Original:   {model1.mse_resid:.4f}")
print(f"MSE Residual Reajustado: {model1_reajustado.mse_resid:.4f}")

--- Comparación de Modelos ---
R² Original:     0.998778
R² Reajustado:   0.998899
MSE Residual Original:   0.3040
MSE Residual Reajustado: 0.2637
